SQL Commands reference notebook
*Co-authored with CoCo*

select -> for selecting columns
with -> for cte
top -> fetch only top n rows
into -> store value of select into variables [Same as oracle]
from -> table which needs to be queried
at -> used for time travel at exact incident [Using timestamp or offset or query_id]
before -> used for time travel just before given incident [Using timestamp or offset or query_id]
changes -> used for tracking changes and retrieve rows that have changed between two points in time
connect by -> used to process hierarchical data in the table.
join -> used to join to tables [skip -> USING, LATERAL keywords]
asof -> used to join time-series data [Nice documentation]
lateral -> [Very very complex; Need to learn]
Match_Recognize -> [Skip]
Pivote -> PIVOT converts one categorical column's values into column headers.
Unpivote -> Unpivote combines multiple columns into single column.
Where -> For filtering rows
Group By -> For grouping
Group By Cube -> For grouping with all possible combination
Group By Grouping Set -> For grouping only specified groups
Group By Rollup -> For heirarchical grouping combination
Having -> Filters rows produced by GROUP BY that do not satisfy a predicate.
Qualify -> QUALIFY clause filters the results of window functions.
Top n, Fetch, Limit -> these are same. used to limit the number of rows.
For Update -> [Skip]

In [ ]:
-- Connect by clause it available in oracle pl/sql
/*
Use CONNECT BY when you need to traverse a hierarchy of unknown depth, such as:
    Employee → Manager hierarchy
    Organization charts
    Product category trees
    Bill of Materials (parent-part → child-part)
    Folder structures

Syntax =>
select columnname from table
start with <condition to idenitfy root>
connect by prior <lower-hierarchy column name> = <higher-hierarchy column name>


Note :-
It can be replaced by self joins, if the hierarchy depth is fixed and known. Else, need to use recursive CTEs.
*/

----------------------- Example 1 -------------------
CREATE OR REPLACE TABLE employees (title VARCHAR, employee_ID INTEGER, manager_ID INTEGER);
INSERT INTO employees (title, employee_ID, manager_ID) VALUES
    ('President', 1, NULL),  -- The President has no manager.
        ('Vice President Engineering', 10, 1),
            ('Programmer', 100, 10),
            ('QA Engineer', 101, 10),
        ('Vice President HR', 20, 1),
            ('Health Insurance Analyst', 200, 20);

select * from employees;

select title, employee_ID, manager_ID, level, CONNECT_BY_ROOT title AS root_title, SYS_CONNECT_BY_PATH(title, ' -> ') from employees
start with manager_ID is NULL -- [OR] title = 'President'
connect by prior employee_ID = manager_ID;

----------------------- Example 2 -------------------
-- The components of a car.
CREATE TABLE components (
    description VARCHAR,
    quantity INTEGER,
    component_ID INTEGER,
    parent_component_ID INTEGER
);

INSERT INTO components (description, quantity, component_ID, parent_component_ID) VALUES
    ('car', 1, 1, 0),
       ('wheel', 4, 11, 1),
          ('tire', 1, 111, 11),
          ('#112 bolt', 5, 112, 11),
          ('brake', 1, 113, 11),
             ('brake pad', 1, 1131, 113),
       ('engine', 1, 12, 1),
          ('piston', 4, 121, 12),
          ('cylinder block', 1, 122, 12),
          ('#112 bolt', 16, 112, 12)   -- Can use same type of bolt in multiple places
    ;
SELECT *  FROM components;

SELECT description, quantity, component_id, parent_component_ID, level, CONNECT_BY_ROOT title AS root_title, SYS_CONNECT_BY_PATH(component_ID, ' -> ') AS path
  FROM components
    START WITH parent_component_ID = 0 -- [OR] component_ID = 1
    CONNECT BY
      parent_component_ID = PRIOR component_ID
  ORDER BY path;

In [ ]:
/*
Cross Join => Every possible combination for tables of both rows
Natural Join => implicit join with matching column name; It doesn't require on condition.
Inner join => join with on condition and return only matching rows
Left Outer Join => join with on condition and return all rows from left table
Right Outer Join => join with on condition and return all rows from right table
Full Outer Join => join with on condition and return rows from both table

Note :- If on condition matches, then only merge will happen.

Skip =>
USING clause with Join
*/

In [ ]:
/*
Definition-1
PIVOT is used to convert row values into columns.
It is usually applied to a column that contains a limited number of distinct values, where each distinct value becomes a separate column.
The values from a selected measure column are then aggregated using functions such as SUM, COUNT, AVG, MAX, or MIN and placed into the corresponding pivoted columns.
PIVOT is mainly used for reporting and data analysis.

Definition-2
PIVOT converts row values into columns.
The distinct values of one column become new columns, and a measure column is aggregated and displayed under those columns.
It is commonly used for reporting and data analysis.

Syntax =>
SELECT *
FROM table_name
PIVOT (
    aggregate_function(measure_column)
    FOR pivot_column
    IN (value_list)
);


SELECT *
FROM table_name
PIVOT (
    aggregate_function(measure_column)
    FOR pivot_column
    IN (<subquery that returns values>)
);


Note :-
In a traditional PIVOT, we usually specify the pivot values explicitly in the IN() clause.
However, Snowflake also supports dynamic pivoting using IN (ANY), which automatically creates columns for all distinct values in the pivot column.

Oracle and Databricks does not support dynamic pivote.
*/

In [ ]:
/*
UNPIVOT does not reverse aggregation performed by PIVOT.
It only converts columns back into rows.
If a PIVOT aggregated multiple rows into one value, the original rows cannot be recovered through UNPIVOT.

UNPIVOT is not always reversible.

-----
PIVOT
Rows -> Columns

UNPIVOT
Columns -> Rows

PIVOT usually decreases the number of rows and increases the number of columns.
UNPIVOT usually increases the number of rows and decreases the number of columns.

-----
Syntax =>
SELECT *
FROM table_name
UNPIVOT (
    value_column
    FOR category_column
    IN (col1, col2, col3)
);

*/

In [ ]:
/*
Group By -> Group-based total
Group By Rollup -> Hierarchy-based subtotals
Group By Cube -> All possible subtotal combinations
Group By Grouping Sets -> User-defined subtotals i.e. Pick exactly the groups you need

ROLLUP generates hierarchical subtotals.
CUBE generates all possible subtotal combinations.
GROUPING SETS allows us to explicitly define which groupings should be generated.

Input Table =>
SELECT region, product, sales FROM sales_table;

1) Group By
SELECT region, product, SUM(sales)
FROM sales_table
GROUP BY region, product;

2) Group By Rollup
SELECT region, product, SUM(sales)
FROM sales_table
GROUP BY ROLLUP(region, product);

Exaplanation :-
It will output 3 grouping combinations.
a) (region, product) -- detail
b) (region) -- subtotal per region
c) () -- grand total

Note :- ROLLUP removes columns from right to left.

3) Group By Cube
SELECT region, product, SUM(sales)
FROM sales_table
GROUP BY CUBE(region, product);

Exaplanation :-
It will output 4 grouping combinations.
a) (region, product) -- subtotal per region per product
b) (region) -- subtotal per region
c) (product) --  subtotal per product
d) () -- grand total


4) Group By Grouping Sets
SELECT region, product, SUM(sales)
FROM sales_table
GROUP BY GROUPING SETS(
    (region, product),
    (region),
    ()
);

Explanation :-
SELECT region, product, SUM(sales)
FROM sales_table
GROUP BY region, product
UNION ALL
SELECT region, NULL AS product, SUM(sales)
FROM sales_table
GROUP BY region
UNION ALL
SELECT NULL AS region, NULL AS product, SUM(sales)
FROM sales_table;


Memory =>
For n columns:
    Rollup creates n+1 grouping combinations [every combination removes columns from right to left]
    CUBE creates 2^n grouping combinations [like subset combination]
    
GROUPING SETS = UNION ALL of multiple GROUP BY queries
GROUP BY GROUPING SETS((a), (b)) is equivalent to GROUP BY a UNION ALL GROUP BY b

*/

In [ ]:
/*
QUALIFY =>
In a SELECT statement, the QUALIFY clause filters the results of window functions.
QUALIFY does with window functions what HAVING does with aggregate functions and GROUP BY clauses.

In the execution order of a query, QUALIFY is therefore evaluated after window functions are computed.
Typically, a SELECT statement’s clauses are evaluated in the order shown below:
FROM
WHERE
GROUP BY
HAVING
WINDOW
QUALIFY
DISTINCT
ORDER BY
LIMIT

*/

-- Create table
CREATE TABLE qt (i INTEGER, p CHAR(1), o INTEGER);
INSERT INTO qt (i, p, o) VALUES
  (1, 'A', 1),
  (2, 'A', 2),
  (3, 'B', 1),
  (4, 'B', 2);

-- Remove duplicates [Old Native Way]
SELECT *
  FROM (
    SELECT i, p, o, ROW_NUMBER() OVER (PARTITION BY p ORDER BY o) AS row_num
      FROM qt)
  WHERE row_num = 1;


-- Remove duplicate [New Way using Qualify]
SELECT i, p, o, ROW_NUMBER() OVER (PARTITION BY p ORDER BY o) AS row_num
  FROM qt
  QUALIFY row_num = 1;
--  OR
SELECT i, p, o
  FROM qt
  QUALIFY ROW_NUMBER() OVER (PARTITION BY p ORDER BY o) = 1;

-- Complex Query
SELECT p, SUM(o) OVER (PARTITION BY p) AS r
  FROM qt
  WHERE o < 4
  GROUP BY p, o
  HAVING SUM(i) > 3
  QUALIFY r IN (
    SELECT MIN(i)
      FROM qt
      GROUP BY p
      HAVING MIN(i) > 3);


In [ ]:
/*
There are three types of parameters:
Account parameters
Session parameters
Object parameters

lINK :- https://docs.snowflake.com/en/sql-reference/parameters


SHOW PARAMETERS
    Generic command

SHOW PARAMETERS IN SESSION
    Current session settings

SHOW PARAMETERS IN USER
    User-level settings

SHOW PARAMETERS IN ACCOUNT
    Account-level settings

SHOW PARAMETERS IN WAREHOUSE
    Warehouse-level settings

SHOW PARAMETERS IN DATABASE
    Database-level settings

Note:-

Parameters are maintained only for specific object types such as accounts, users, sessions, warehouses, databases, schemas, tables, and tasks.
Objects like file formats, stages, functions, and procedures are configured through object properties rather than parameter settings.
*/

SHOW PARAMETERS; -- 159
SHOW PARAMETERS IN SESSION; -- 159
SHOW PARAMETERS IN ACCOUNT; -- 295
SHOW PARAMETERS IN USER RAJMAURYA639366; -- 178
SHOW PARAMETERS IN WAREHOUSE COMPUTE_WH; -- 5
SHOW PARAMETERS IN DATABASE our_first_db; -- 43

-- Alter session parameters
ALTER SESSION SET QUERY_TAG = 'ETL_JOB';
ALTER SESSION SET TIMEZONE = 'UTC';

-- Alter database parameters
ALTER DATABASE sales_db
SET DATA_RETENTION_TIME_IN_DAYS = 7;

## Old Oracle Join Syntax vs New (ANSI) Join Syntax

Comma represents cartesian join. `(+)` represents Oracle's old outer join operator. It represents the **optional** table.

---

### Inner Join

**Old Syntax:**
```sql
SELECT *
FROM emp e, dept d
WHERE e.deptno = d.deptno;
```

**New Syntax:**
```sql
SELECT *
FROM emp e
INNER JOIN dept d
    ON e.deptno = d.deptno;
```

---

### Left Outer Join

**Old Syntax:**
```sql
SELECT *
FROM emp e, dept d
WHERE e.deptno = d.deptno(+);
```

**New Syntax:**
```sql
SELECT *
FROM emp e
LEFT OUTER JOIN dept d
    ON e.deptno = d.deptno;
```

---

### Right Outer Join

**Old Syntax:**
```sql
SELECT *
FROM emp e, dept d
WHERE e.deptno(+) = d.deptno;
```

**New Syntax:**
```sql
SELECT *
FROM emp e
RIGHT OUTER JOIN dept d
    ON e.deptno = d.deptno;
```

---

### Quick Conversion Rules

| # | Old Oracle Syntax | New ANSI Syntax |
|---|---|---|
| A | `FROM A, B WHERE A.id = B.id` | `FROM A INNER JOIN B ON A.id = B.id` |
| B | `FROM A, B WHERE A.id = B.id(+)` | `FROM A LEFT JOIN B ON A.id = B.id` |
| C | `FROM A, B WHERE A.id(+) = B.id` | `FROM A RIGHT JOIN B ON A.id = B.id` |

---

### Memory Trick

| `(+)` Position | Join Type |
|---|---|
| No `(+)` | INNER JOIN (if join condition exists) |
| Right side `(+)` | LEFT JOIN |
| Left side `(+)` | RIGHT JOIN |
| Both sides `(+)` | Invalid (Not supported in Oracle) |

Q. Does Snowflake provide services like disaster recovery? It is not covered with Databricks.
Ans. Yes, Snowflake provides disaster recovery services. It offers features such as immutable backups, failover groups, and replication to ensure data integrity and quick recovery from potential failures


Note :- Snowflake supports multi-table inserts similar to Oracle, though the syntax and features are not identical.
INSERT ALL => For each source row, execute all INTO clauses.
INSERT ALL with condition => For each source row, execute all WHEN blocks whose conditions are TRUE.
INSERT FIRST with condition => For each source row, execute only the first matching WHEN block. All INTO clauses inside that matching block are executed.



### CREATE OR REPLACE FUNCTION (SQL user-defined function)
CREATE OR REPLACE PROCEDURE (Snowflake Scripting stored procedure)

Types of tables =>
Normal Tables
Dynamic Tables
Event Tables
Iceberg Tables [CREATE ICEBERG TABLE ]
Transient Tables
Temporary Tables
Hybrid Tables


-- Retrieve the DDL for the source schema.
SELECT GET_DDL ('schema', 'source', true);

-- Describe Command
DESC RESULT LAST_QUERY_ID();
DESC RESULT '<query-id>';
DESC TRANSACTION <transaction_id>;

Q. query_id vs transaction_id in snowflake?
Query_id :- A Query ID uniquely identifies a single SQL statement execution.
Transaction_id :- A Transaction ID identifies an entire transaction.

### Creating Transaction =>
-- TRANSACTION-1
BEGIN;
-- QUERY-1
-- QUERU-2
-- QUERY-3
COMMIT;

-- TRANSACTION-2
BEGIN;
-- QUERY-1
-- QUERU-2
-- QUERY-3
ROLLBACK;

Important Rules
1. Transaction starts with BEGIN
2. Ends with either COMMIT; or ROLLBACK;
3. Nested Transactions are not supported. One active transaction at a time.
4. By default:
    AUTOCOMMIT = TRUE
So: Any sql statement that executes outside transaction block, gets automatically committed.
5. If you need multiple statements treated as one unit, use transactoin block.
BEGIN;
    -- QUERY-1
    -- QUERY-2
COMMIT;

When to use transaction_block and when to write standalone sql statements?
Ans. When we want to execute multiple sql statements as single unit, then enclose them insdie transaction block.
Otherwise, write them as standalone sql statement.

NOTE:- First transaction [TRANSACTION-1] will commit, only if all queries run successfully. If any of them will fail, it will perform rollback.
Second transaction [TRANSACTION-2] will always rollback. It is good for unit testing or dry run where we don't want to commit the changes.
Each transaction gets unique transaction_id. Within each transaction, each query also gets unique query_id.

| Query ID                                                     | Transaction ID                                |
| ------------------------------------------------------------ | --------------------------------------------- |
| Identifies a single SQL statement                            | Identifies a transaction                      |
| Generated for every query                                    | Generated for every transaction               |
| Many Query IDs can belong to one Transaction ID              | One Transaction ID can contain many Query IDs |
| Used in Query History, RESULT\_SCAN, Time Travel (STATEMENT) | Used for transaction tracking and auditing    |


Alternative ways to create transaction block =>
-- Syntax-1
BEGIN WORK;
    select 23;
COMMIT WORK;

-- Syntax-2
BEGIN;
    select 23;
COMMIT;

-- Syntax-3
BEGIN TRANSACTION;
    select 23;
COMMIT;